In [2]:
import ee

ee.Authenticate()

ee.Initialize(project= 'induswater')

Features which are geometric objects with a list of properties. For example, a watershed with some properties such as name and area, is an ee.Feature.
Images which are like features, but may include several bands. For example, the ground elevation given by the USGS here is an ee.Image.
Collections which are groups of features or images. For example, the Global Administrative Unit Layers giving administrative boundaries is a ee.FeatureCollection and the MODIS Land Surface Temperature dataset is an ee.ImageCollection.

In [3]:
i_date='1999-01-01'
m_date='2012-06-06'
f_date='2025-12-12'

dataset1 = ee.ImageCollection('NASA/GPM_L3/IMERG_V07').filterDate(i_date, m_date)
dataset2 = ee.ImageCollection('NASA/GPM_L3/IMERG_V07').filterDate(m_date, f_date)


print(f"Date range: {i_date} to {f_date}")
print(f"Number of images: {dataset1.size().getInfo()}")

Date range: 1999-01-01 to 2025-12-12
Number of images: 235440


In [ ]:
import geemap
Map=geemap.Map()
#adding shapefile first
ee_object = geemap.shp_to_ee('indus_shapefile/upperindusbd.shp')
Map.addLayer(ee_object, {'color': 'red'}, 'Upper indus basin')

#precipitation layer
precipitation = dataset1.select('precipitation').max()
# mask = precipitation.gt(0.5)
# precipitation = precipitation.updateMask(mask) #good for visualization but dont use for rest pipeline as it will set null to the values that fail to meet the criteria

palette = [
  '000096','0064ff', '00b4ff', '33db80', '9beb4a',
  'ffeb00', 'ffb300', 'ff6400', 'eb1e00', 'af0000'
]
precipitationVis = {'min': 0, 'max': 15, 'palette': palette}
Map.addLayer(precipitation, precipitationVis, 'Precipitation (mm/hr)')
Map.centerObject(ee_object, 7)


In [ ]:
Map.save('map_output.html')
print('map saved successfully')

In [ ]:
Map

data is ready to be downloaded and accurate too

In [4]:
#function to download
def calculate_monthly_precip(image):
    date = image.date().format('YYYY-MM-dd')

    mean_val = image.reduceRegion(
        reducer= ee.Reducer.mean(),
        geometry= ee_object.geometry(),
        scale= 11132
    ).get('precipitation')

    return ee.Feature(None, {'date': date, 'precip_mm_hr': mean_val})




In [ ]:
import pandas as pd
monthly_stats= dataset1.map(calculate_monthly_precip).getInfo()
data= [f['properties'] for f in monthly_stats['features']]
df = pd.DataFrame(data)

df.to_csv('precipitation_2000_2025.csv', index=False)
print('all done sire')

In [ ]:
i_date='1999-01-01'
m_date='2012-06-01'
f_date='2026-01-01'

dataset1 = ee.ImageCollection('NASA/GPM_L3/IMERG_V07').filterDate(i_date, '2012-06-01')
dataset2 = ee.ImageCollection('NASA/GPM_L3/IMERG_V07').filterDate('2012-06-01', f_date)


print(f"Date range: {i_date} to {f_date}")
print(f"total images printed:{dataset1.size().getInfo()}")

Date range: 1999-01-01 to 2025-12-12
total images printed:235200


In [ ]:
#define download function
def get_daily_precip(image):
    date= image.date().format('YYYY-MM-dd')

    mean_val = image.reduceRegion(
        reducer = ee.Reducer.mean(), # i think this mean is for the region's mean and not monhtly mean
        geometry = ee_object.geometry(),
        scale = 11132
    ).get('precipitation')

    return ee.Feature(None, {'date': date, 'precip_mm_hr': mean_val})


.getInfo() triggers computation - This is when Earth Engine actually processes your request
It's synchronous - Your code waits for the result

Earth Engine (Server)          Python (Your Computer)
─────────────────────          ─────────────────────
ee.FeatureCollection           .getInfo()
  ├─ Feature 1                  ──────►           dict
  ├─ Feature 2                                    {'type': 'FeatureCollection',
  └─ Feature 3                                     'features': [
                                                     {'properties': {...}},
                                                     {'properties': {...}}
                                                   ]}
meaning you can't access, process download without .getInfo

In [15]:
import pandas as pd 
datasets=[dataset1] #divide to prevent hitting limit
dfs=[]
for d in datasets:
    raw_daily_values = d.map(get_daily_precip).getInfo()
    data=[f['properties'] for f in raw_daily_values['features']]
    dfs.append(pd.DataFrame(data))

final_df = pd.concat(dfs, ignore_index=True)
final_df.to_csv("precipitation_daily_2000_2025.csv")


EEException: Collection query aborted after accumulating over 5000 elements.

In [ ]:
import ee
import pandas as pd
import geemap

ee.Initialize(project='induswater')

ee_object = geemap.shp_to_ee('indus_shapefile/upperindusbd.shp')

# Function to aggregate by month
def calculate_monthly_mean(year, month, day):
    start = ee.Date.fromYMD(year, month, 1)
    end = start.advance(1, 'month')
    
    monthly_collection = ee.ImageCollection('NASA/GPM_L3/IMERG_V07') \
        .filterDate(start, end) \
        .select('precipitation')
    
    # Calculate mean for the month
    monthly_mean = monthly_collection.mean()
    
    mean_val = monthly_mean.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=ee_object.geometry(),
        scale=11132,
        maxPixels=1e9
    ).get('precipitation')
    
    return {
        'year': year,
        'month': month,
        'date': f'{year}-{month:02d}',
        'precip_mm_hr': mean_val
    }

# Generate monthly data (only ~300 records for 25 years!)
data = []
for year in range(2000, 2025):
    print(f"Processing {year}...")
    for month in range(1, 13):
        try:
            result = calculate_monthly_mean(year, month)
            # Convert to Python immediately
            result['precip_mm_hr'] = result['precip_mm_hr'].getInfo()
            data.append(result)
        except Exception as e:
            print(f"  Error for {year}-{month:02d}: {e}")

df = pd.DataFrame(data)
df.to_csv('precipitation_monthly_2000_2025.csv', index=False)
print(f"✅ Done! {len(df)} monthly records saved")

Processing 2000...
Processing 2001...
Processing 2002...
Processing 2003...
Processing 2004...
Processing 2005...
Processing 2006...
Processing 2007...
Processing 2008...
Processing 2009...
Processing 2010...
Processing 2011...
Processing 2012...
Processing 2013...
Processing 2014...
Processing 2015...
Processing 2016...
Processing 2017...
Processing 2018...
Processing 2019...
Processing 2020...
Processing 2021...
Processing 2022...
Processing 2023...
Processing 2024...
✅ Done! 300 monthly records saved


In [18]:
import ee
import pandas as pd
import geemap
from datetime import datetime, timedelta

ee.Initialize(project='induswater')

ee_object = geemap.shp_to_ee('indus_shapefile/upperindusbd.shp')

# Define date range
start_date = datetime(2000, 6, 1)
end_date = datetime(2026, 1, 1)

# Build all daily computations first (don't download yet)
print("Building daily computations...")
daily_features = []

current_date = start_date
while current_date < end_date:
    date_str = current_date.strftime('%Y-%m-%d') #to convert datetime format to a string, why not string only already? so that we could loop over
    
    # Create Earth Engine dates
    ee_start = ee.Date(date_str)
    ee_end = ee_start.advance(1, 'day')
    
    # Get all 48 half-hourly images for this day
    daily_collection = ee.ImageCollection('NASA/GPM_L3/IMERG_V07') \
        .filterDate(ee_start, ee_end) \
        .select('precipitation')
    
    # Aggregate 48 images into 1 daily mean (on server)
    daily_mean = daily_collection.mean()
    
    # Calculate basin-wide mean
    mean_val = daily_mean.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=ee_object.geometry(),
        scale=11132,
        maxPixels=1e9
    ).get('precipitation')
    
    # Create feature (still on server, not downloaded)
    feature = ee.Feature(None, {
        'date': date_str,
        'precip_mm_hr': mean_val
    })
    daily_features.append(feature)
    
    current_date += timedelta(days=1) #in datetime library, this is how we move date by one day

print(f"Built {len(daily_features)} daily computations")

# Download all at once (ONE request for ~9,000 days)
print("Downloading all daily data in one batch...")
daily_collection = ee.FeatureCollection(daily_features)
all_data = daily_collection.getInfo()

# Convert to DataFrame
print("Converting to DataFrame...")
data = [f['properties'] for f in all_data['features']]
df = pd.DataFrame(data)

# Save to CSV
df.to_csv('precipitation_daily_2000_2025.csv', index=False)
print(f"✅ Done! {len(df)} daily records saved")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")

Building daily computations...
Built 9345 daily computations


EEException: Request payload size exceeds the limit: 10485760 bytes.

In [ ]:
import ee
import pandas as pd
import geemap

ee.Initialize(project='induswater')
ee_object = geemap.shp_to_ee('indus_shapefile/upperindusbd.shp')

# Process year by year
all_data = []

# date.advance(amount, unit)
for year in range(2024, 2025):
    print(f"Processing {year}...")
    
    # Define year dates
    if year == 2000:
        year_start = ee.Date('2000-06-01')
    else:
        year_start = ee.Date(f'{year}-01-01')
    
    if year == 2024:
        year_end = ee.Date('2026-01-01')
    else:
        year_end = ee.Date(f'{year+1}-01-01')
    
    # Create daily date sequence for this year
    num_days = year_end.difference(year_start, 'day').ceil()
    daily_dates = ee.List.sequence(0, num_days.subtract(1)).map(
        lambda d: year_start.advance(d, 'day')
    )
    
    # Function to process one day
    def process_day(date):
        date = ee.Date(date)
        end = date.advance(1, 'day')
        
        daily_collection = ee.ImageCollection('NASA/GPM_L3/IMERG_V07') \
            .filterDate(date, end) \
            .select('precipitation')
        
        daily_mean = daily_collection.mean()
        
        mean_val = daily_mean.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=ee_object.geometry(),
            scale=11132,
            maxPixels=1e9
        ).get('precipitation')
        
        return ee.Feature(None, {
            'date': date.format('YYYY-MM-dd'),
            'precip_mm_hr': mean_val
        })
    
    # Process all days in this year
    daily_features = daily_dates.map(process_day)
    daily_collection = ee.FeatureCollection(daily_features)
    
    # Download this year's data
    year_data = daily_collection.getInfo()
    year_records = [f['properties'] for f in year_data['features']]
    all_data.extend(year_records)
    
    print(f"  ✅ {year}: {len(year_records)} days")

# Combine all years
df = pd.DataFrame(all_data)
df.to_csv('precipitation_daily_2000_2025.csv', index=False)
print(f"\n✅ Complete! {len(df)} daily records saved")

Processing 2000...
  ✅ 2000: 214 days
Processing 2001...
  ✅ 2001: 365 days
Processing 2002...
  ✅ 2002: 365 days
Processing 2003...
  ✅ 2003: 365 days
Processing 2004...
  ✅ 2004: 366 days
Processing 2005...
  ✅ 2005: 365 days
Processing 2006...
  ✅ 2006: 365 days
Processing 2007...
  ✅ 2007: 365 days
Processing 2008...
  ✅ 2008: 366 days
Processing 2009...
  ✅ 2009: 365 days
Processing 2010...
  ✅ 2010: 365 days
Processing 2011...
  ✅ 2011: 365 days
Processing 2012...
  ✅ 2012: 366 days
Processing 2013...
  ✅ 2013: 365 days
Processing 2014...
  ✅ 2014: 365 days
Processing 2015...
  ✅ 2015: 365 days
Processing 2016...
  ✅ 2016: 366 days
Processing 2017...
  ✅ 2017: 365 days
Processing 2018...
  ✅ 2018: 365 days
Processing 2019...
  ✅ 2019: 365 days
Processing 2020...
  ✅ 2020: 366 days
Processing 2021...
  ✅ 2021: 365 days
Processing 2022...
  ✅ 2022: 365 days
Processing 2023...
  ✅ 2023: 365 days
Processing 2024...
  ✅ 2024: 395 days

✅ Complete! 9009 daily records saved


# snow

In [24]:
dataset_snow = ee.ImageCollection('MODIS/061/MOD10A1') \
    .filter(ee.Filter.date('2024-01-01','2024-04-30'))

snowcover=dataset_snow.select('NDSI_Snow_Cover')



In [27]:
snowCoverVis = {
  'min': 0.0,
  'max': 100.0,
  'palette': ['black', '0dffff', '0524ff', 'ffffff'],
}

In [34]:
import geemap
Map= geemap.Map()
ee_object= geemap.shp_to_ee('indus_shapefile/upperindusbd.shp')
Map.addLayer(ee_object, {'color':'red'}, 'upper_basin')
Map.setCenter(-41.13, 76.35, 3)
Map.addLayer(snowcover, snowCoverVis, 'Snow Cover')

In [35]:
Map

Map(center=[76.35, -41.13], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

In [ ]:
def mask_clouds_and_ice(image):
    # The MOD10A1 product has a band called 'NDSI_Snow_Cover_Basic_QA'
    qa = image.select('NDSI_Snow_Cover_Basic_QA')
    
    # We use bitwise operations to isolate bits 0-15 (Quality flags)
    # Bit 0-1: 0 = Best, 1 = Good, 2 = OK
    mask = qa.bitwiseAnd(1 << 0).eq(0).And(qa.bitwiseAnd(1 << 1).eq(0))
    
    # Update the image to only keep the 'Clear Sky' high-quality pixels
    return image.updateMask(mask)

# Apply it to your MODIS collection
modis_clean = ee.ImageCollection("MODIS/061/MOD10A1") \
                .filterBounds(uib_ee) \
                .map(mask_clouds_and_ice)

In [ ]:
dataset_snow = ee.ImageCollection('MODIS/061/MOD10A1') \
    .filter(ee.Filter.date('2000-02-24','2026-01-01'))
snowcover = dataset_snow.select('NDSI_Snow_Cover')


In [46]:
def bitwiseExtract(input, from_bit, to_bit):
    maskSize= ee.Number(1).add(to_bit).subtract(from_bit)
    mask = ee.Number(1).leftShift(maskSize).subtract(1)
    return input.rightShift(from_bit).bitwiseAnd(mask)

#we use this function when using multiple bits or for easier logic understanding. The standard GEE way is used in the next cell and in this pipeline.


In [ ]:
def modisMask(image):
    qa_sca_basic = image.select('NDSI_Snow_Cover_Basic_QA')
    qa_sca = image.select('NDSI_Snow_Cover_Algorithm_Flags_QA') 
    mask= qa_sca_basic.lte(1)

    inland_mask=qa_sca.bitwiseAnd(1<<0).eq(0)
    mask = mask.And(inland_mask)

    zenith_mask= qa_sca.bitwiseAnd(1<<7).eq(0)
    mask = mask.And(zenith_mask)

    return image.select('NDSI_Snow_Cover').updateMask(mask)

In [49]:
masked_snowcover = dataset_snow.map(modisMask)

In [ ]:
def calculate_sca(image):
    binary_snow = image.gt(40) 
    
    # 2. Get the area of one pixel
    pixel_area = ee.Image.pixelArea()
    
    # 3. Multiply mask by area to get a map of 'Square Meters per Snow Pixel'
    snow_area_image = binary_snow.multiply(pixel_area)
    
    # 4. Use SUM to add up all those square meters
    stats = snow_area_image.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=ee_object.geometry(),
        scale=500, # MODIS resolution is 500m
        maxPixels=1e9
    )
    
    # 5. Convert to square kilometers (divide by 1,000,000)
    sca_km2 = ee.Number(stats.get('NDSI_Snow_Cover')).divide(1e6)
    
    return image.set('SCA_km2', sca_km2)

In [ ]:
import geemap
import time
import datetime
import pandas as pd
import ee
overall_sca_data=[]
#year by year
for year in range(2000,2026):
    if (year==2000):
        start_date=ee.Date('2000-02-24')
    else:
        start_date=ee.Date(f'{year}-01-01')

    if (year==2025):
        end_date= ee.Date('2026-01-01')
    else:
        end_date=ee.Date(f'{year+1}-01-01')

    num_days=end_date.difference(start_date, 'day').ceil()
    daily_dates = ee.List.sequence(0, num_days.subtract(1)).map(lambda d: start_date.advance(d, 'day'))

    def process_day(date):
        date=ee.Date(date)
        end= date.advance(1, 'day')

        daily_collection= masked_snowcover \
            .filterDate(date, end) \
            .select('NDSI_Snow_Cover') \
            
        daily_image = daily_collection.first()

            
        binary_snow= daily_image.gt(40)
        pixel_area= ee.Image.pixelArea()
        snow_area_image= binary_snow.multiply(pixel_area)

        stats = snow_area_image.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=ee_object.geometry(),
        scale=500, # MODIS resolution is 500m
        maxPixels=1e9
        )
        sca_km2 = ee.Number(stats.get('NDSI_Snow_Cover')).divide(1e6)
            
        return ee.Feature(None, {
            'date': date.format('YYYY-MM-dd'),
            'sca_km2': sca_km2
        })

    daily_features=daily_dates.map(process_day)
    daily_collection= ee.FeatureCollection(daily_features)

    #download the date
    year_data = daily_collection.getInfo()
    year_records = [f['properties'] for f in year_data['features']]
    overall_sca_data.extend(year_records)

    print(f'{year}, {len(year_records)} number of days')
    time.sleep(5)


#finally we combine to a single dataframe

df=pd.DataFrame(overall_sca_data)
df.to_csv('sca_2000_2025.csv', index=False)
print(f'{len(df)} saved')

EEException: Too many concurrent aggregations.

In [65]:
import ee

# 1. Define the whole range as a single list (Server-side)
start_total = ee.Date('2000-02-24')
end_total = ee.Date('2026-01-01')
num_days = end_total.difference(start_total, 'day')
daily_dates = ee.List.sequence(0, num_days.subtract(1)).map(lambda d: start_total.advance(d, 'day'))

def process_day(date):
    date = ee.Date(date)
    # Filter for the day and check if an image exists
    daily_collection = masked_snowcover.filterDate(date, date.advance(1, 'day'))
    img_count = daily_collection.size()
    
    # Use ee.Algorithms.If to handle cloudy/missing days safely
    def calculate():
        daily_image = ee.Image(daily_collection.first())
        binary_snow = daily_image.gt(40)
        pixel_area = ee.Image.pixelArea()
        snow_area_image = binary_snow.multiply(pixel_area)
        
        stats = snow_area_image.reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=ee_object.geometry(),
            scale=500, # Increased to 1km to avoid memory errors for huge UIB area
            maxPixels=1e9
        )
        return ee.Number(stats.get('NDSI_Snow_Cover')).divide(1e6)

    # If image count > 0, calculate; else return -9999 (standard NoData value)
    sca_km2 = ee.Number(ee.Algorithms.If(img_count.gt(0), calculate(), -9999))
        
    return ee.Feature(None, {
        'date': date.format('YYYY-MM-dd'),
        'sca_km2': sca_km2
    })

# Map the function over the ENTIRE 25-year list at once
uib_daily_features = daily_dates.map(process_day)
final_fc = ee.FeatureCollection(uib_daily_features)

# 2. EXPORT: This is the key. It bypasses the "Concurrent Aggregation" limit.
task = ee.batch.Export.table.toDrive(
    collection=final_fc,
    description='UIB_SCA_Daily_25Year_with500mscale',
    fileFormat='CSV',
    selectors=['date', 'sca_km2']
)

task.start()
print("Task started! Go to code.earthengine.google.com and check the 'Tasks' tab on the right.")

Task started! Go to code.earthengine.google.com and check the 'Tasks' tab on the right.


In [ ]:
import geemap
import time
import datetime
import pandas as pd
import ee
overall_sca_data=[]
#year by year
for year in range(2000,2026):
    if (year==2000):
        start_date=ee.Date('2000-02-24')
    else:
        start_date=ee.Date(f'{year}-01-01')

    if (year==2025):
        end_date= ee.Date('2026-01-01')
    else:
        end_date=ee.Date(f'{year+1}-01-01')

    num_days=end_date.difference(start_date, 'day').ceil()
    daily_dates = ee.List.sequence(0, num_days.subtract(1)).map(lambda d: start_date.advance(d, 'day'))

    def process_day(date):
        date=ee.Date(date)
        end= date.advance(1, 'day')

        daily_collection= masked_snowcover \
            .filterDate(date, end) \
            .select('NDSI_Snow_Cover') \
            
        daily_image = daily_collection.first()

            
        binary_snow= daily_image.gt(40)
        pixel_area= ee.Image.pixelArea()
        snow_area_image= binary_snow.multiply(pixel_area)

        stats = snow_area_image.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=ee_object.geometry(),
        scale=500, # MODIS resolution is 500m
        maxPixels=1e9
        )
        sca_km2 = ee.Number(stats.get('NDSI_Snow_Cover')).divide(1e6)
            
        return ee.Feature(None, {
            'date': date.format('YYYY-MM-dd'),
            'sca_km2': sca_km2
        })

    daily_features=daily_dates.map(process_day)
    daily_collection= ee.FeatureCollection(daily_features)

    #download the date
    year_data = daily_collection.getInfo()
    year_records = [f['properties'] for f in year_data['features']]
    overall_sca_data.extend(year_records)

    print(f'{year}, {len(year_records)} number of days')
    time.sleep(5)


#finally we combine to a single dataframe

df=pd.DataFrame(overall_sca_data)
df.to_csv('sca_2000_2025.csv', index=False)
print(f'{len(df)} saved')

In [69]:
import ee
import pandas
import datetime
temperature_range=[]

for year in range(2000,2026):
    start_date = ee.Date(f'{year}-01-01')
    if (year==2025):
        end_date = ee.Date('2026-01-01')
    else:
        end_date= ee.Date(f'{year+1}-01-01')

    num_days = end_date.difference(start_date, 'day').ceil()
    daily_dates = ee.List.sequence(0, num_days.subtract(1)).map(lambda d: start_date.advance(d, 'day'))

    def process_day(date):
        date = ee.Date(date)
        end = date.advance(1, 'day')

        hourly_temp = ee.ImageCollection('ECMWF/ERA5_LAND/HOURLY') \
            .filterDate(date, end) \
            .select('temperature_2m')
        
        #we now have all the temps for the day, we need mean, min, max

        daily_stats= hourly_temp.reduce(
            ee.Reducer.mean().combine(
                reducer2= ee.Reducer.min().combine(
                    reducer2=ee.Reducer.max(),
                    sharedInputs=True
                ),
                sharedInputs=True,
            )
        )

        #now over to the reduceRegion
        regional_stats = daily_stats.reduceRegion(
            reducer= ee.Reducer.mean(),
            geometry= ee_object.geometry(),
            scale=11132
        )

        return ee.Feature(None,{
            'date': date.format('YYYY-MM-dd'),
            'temp_mean': regional_stats.get('temperature_2m_mean'),
            'max_temp' : regional_stats.get('temperature_2m_max'),
            'min_temp' : regional_stats.get('temperature_2m_min')
        })
    
    daily_features=daily_dates.map(process_day)
    daily_collection= ee.FeatureCollection(daily_features)

    year_data= daily_collection.getInfo()
    year_records= [f['properties'] for f in year_data['features']]
    temperature_range.extend(year_records)

    print(f'{year}, {len(year_records)} number of days')


#finally we combine to a single dataframe

df=pd.DataFrame(temperature_range)
df.to_csv('temperature2000_2025.csv', index=False)
print(f'{len(df)} saved')
    

        


2000, 366 number of days
2001, 365 number of days
2002, 365 number of days
2003, 365 number of days
2004, 366 number of days
2005, 365 number of days
2006, 365 number of days
2007, 365 number of days
2008, 366 number of days
2009, 365 number of days
2010, 365 number of days
2011, 365 number of days
2012, 366 number of days
2013, 365 number of days
2014, 365 number of days
2015, 365 number of days
2016, 366 number of days
2017, 365 number of days
2018, 365 number of days
2019, 365 number of days
2020, 366 number of days
2021, 365 number of days
2022, 365 number of days
2023, 365 number of days
2024, 366 number of days
2025, 365 number of days
9497 saved


In [70]:
import ee
import pandas
import datetime
temperature_range=[]

for year in range(2000,2026):
    start_date = ee.Date(f'{year}-01-01')
    if (year==2025):
        end_date = ee.Date('2026-01-01')
    else:
        end_date= ee.Date(f'{year+1}-01-01')

    num_days = end_date.difference(start_date, 'day').ceil()
    daily_dates = ee.List.sequence(0, num_days.subtract(1)).map(lambda d: start_date.advance(d, 'day'))

    def process_day(date):
        date = ee.Date(date)
        end = date.advance(1, 'day')

        hourly_temp = ee.ImageCollection('ECMWF/ERA5_LAND/HOURLY') \
            .filterDate(date, end) \
            .select('surface_net_solar_radiation')
        
        #we now have all the temps for the day, we need mean, min, max

        # daily_stats= hourly_temp.reduce(
        #     ee.Reducer.mean().combine(
        #         reducer2= ee.Reducer.min().combine(
        #             reducer2=ee.Reducer.max(),
        #             sharedInputs=True
        #         ),
        #         sharedInputs=True,
        #     )
        # )
        daily_stats= hourly_temp.sum()

        #now over to the reduceRegion
        regional_stats = daily_stats.reduceRegion(
            reducer= ee.Reducer.mean(),
            geometry= ee_object.geometry(),
            scale=11132
        )

        return ee.Feature(None,{
            'date': date.format('YYYY-MM-dd'),
            'surface_net_solar_radiation_J/m²': regional_stats.get('surface_net_solar_radiation')
        })
    
    daily_features=daily_dates.map(process_day)
    daily_collection= ee.FeatureCollection(daily_features)

    year_data= daily_collection.getInfo()
    year_records= [f['properties'] for f in year_data['features']]
    temperature_range.extend(year_records)

    print(f'{year}, {len(year_records)} number of days')


#finally we combine to a single dataframe

df=pd.DataFrame(temperature_range)
df.to_csv('temperature2000_2025.csv', index=False)
print(f'{len(df)} saved')
    

        


2000, 366 number of days
2001, 365 number of days
2002, 365 number of days
2003, 365 number of days
2004, 366 number of days
2005, 365 number of days
2006, 365 number of days
2007, 365 number of days
2008, 366 number of days
2009, 365 number of days
2010, 365 number of days
2011, 365 number of days
2012, 366 number of days
2013, 365 number of days
2014, 365 number of days
2015, 365 number of days
2016, 366 number of days
2017, 365 number of days
2018, 365 number of days
2019, 365 number of days
2020, 366 number of days
2021, 365 number of days
2022, 365 number of days
2023, 365 number of days
2024, 366 number of days
2025, 365 number of days
9497 saved


In [73]:
import ee
import pandas
import datetime
temperature_range=[]

for year in range(2000,2026):
    start_date = ee.Date(f'{year}-01-01')
    if (year==2025):
        end_date = ee.Date('2026-01-01')
    else:
        end_date= ee.Date(f'{year+1}-01-01')

    num_days = end_date.difference(start_date, 'day').ceil()
    daily_dates = ee.List.sequence(0, num_days.subtract(1)).map(lambda d: start_date.advance(d, 'day'))

    def process_day(date):
        date = ee.Date(date)
        end = date.advance(1, 'day')

        hourly_temp = ee.ImageCollection('ECMWF/ERA5_LAND/HOURLY') \
            .filterDate(date, end) \
            .select('dewpoint_temperature_2m')
        
        #we now have all the temps for the day, we need mean, min, max

        daily_stats= hourly_temp.reduce(
            ee.Reducer.mean().combine(
                reducer2= ee.Reducer.min().combine(
                    reducer2=ee.Reducer.max(),
                    sharedInputs=True
                ),
                sharedInputs=True,
            )
        )
        # daily_stats= hourly_temp.mean()

        #now over to the reduceRegion
        regional_stats = daily_stats.reduceRegion(
            reducer= ee.Reducer.mean(),
            geometry= ee_object.geometry(),
            scale=11132
        )

        return ee.Feature(None,{
            'date': date.format('YYYY-MM-dd'),
            'mean_dew': regional_stats.get('dewpoint_temperature_2m_mean'),
            'max_dew' : regional_stats.get('dewpoint_temperature_2m_max'),
            'min_dew' : regional_stats.get('dewpoint_temperature_2m_min')
        })
    
    daily_features=daily_dates.map(process_day)
    daily_collection= ee.FeatureCollection(daily_features)

    year_data= daily_collection.getInfo()
    year_records= [f['properties'] for f in year_data['features']]
    temperature_range.extend(year_records)

    print(f'{year}, {len(year_records)} number of days')


#finally we combine to a single dataframe

df=pd.DataFrame(temperature_range)
df.to_csv('dewpoint_temp.csv', index=False)
print(f'{len(df)} saved')
    

        


2000, 366 number of days
2001, 365 number of days
2002, 365 number of days
2003, 365 number of days
2004, 366 number of days
2005, 365 number of days
2006, 365 number of days
2007, 365 number of days
2008, 366 number of days
2009, 365 number of days
2010, 365 number of days
2011, 365 number of days
2012, 366 number of days
2013, 365 number of days
2014, 365 number of days
2015, 365 number of days
2016, 366 number of days
2017, 365 number of days
2018, 365 number of days
2019, 365 number of days
2020, 366 number of days
2021, 365 number of days
2022, 365 number of days
2023, 365 number of days
2024, 366 number of days
2025, 365 number of days
9497 saved


In [8]:
def bitwiseExtract(input, from_bit, to_bit):
    maskSize= ee.Number(1).add(to_bit).subtract(from_bit)
    mask = ee.Number(1).leftShift(maskSize).subtract(1)
    return input.rightShift(from_bit).bitwiseAnd(mask)

#we use this function when using multiple bits or for easier logic understanding. The standard GEE way is used in the next cell and in this pipeline.


In [9]:
def maskMODISNDVI(image):
    qa_band= image.select('DetailedQA')
    mask=bitwiseExtract(qa_band, 0, 1).eq(0).And(bitwiseExtract(qa_band, 2,5).lte(2))
    # adjacent_cloud_band=qa_band.bitwiseAnd(1<<8).eq(0)
    # mask=mask.And(adjacent_cloud_band)
    ndvi = image.select('NDVI')
    valid_range_mask = ndvi.gt(-2000) # Excludes the -3000 fill value
    
    final_mask = mask.And(valid_range_mask)
    
    return ndvi.updateMask(final_mask).multiply(0.0001)
    ## Bit 15: Possible shadow- to be considered later maybe
    # shadow_mask = qa.bitwiseAnd(1<<15).eq(0)
    # mask = mask.And(shadow_mask)

In [10]:
#load the dataset band(normal image, maybe)
ndvi_dataset= ee.ImageCollection('MODIS/061/MOD13Q1') \
            .filterDate('2000-02-18','2026-01-01') \
            .select(['NDVI','DetailedQA'])

#apply mask to the entire dataset

ndvi_masked= ndvi_dataset.map(maskMODISNDVI)


In [17]:
import ee
import pandas
import datetime
temperature_range=[]

for year in range(2000,2026):
    start_date = ee.Date(f'{year}-01-01')
    if (year==2025):
        end_date = ee.Date('2026-01-01')
    else:
        end_date= ee.Date(f'{year+1}-01-01')

    num_days = end_date.difference(start_date, 'day').ceil()
    daily_dates = ee.List.sequence(0, num_days.subtract(1)).map(lambda d: start_date.advance(d, 'day'))

    def process_day(date):
        date = ee.Date(date)
        end = date.advance(1, 'day')

        hourly_temp = ndvi_masked \
            .filterDate(date, end) \
            .select('NDVI')
        
        # Check if collection has images
        size = hourly_temp.size()
        
        def compute():
            daily_stats = hourly_temp.median()
            regional_stats = daily_stats.reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=ee_object.geometry(),
                scale=250,
                maxPixels=1e9
            )
            # Safe get with unmask
            return ee.Number(ee.Algorithms.If(
                regional_stats.contains('NDVI'),
                regional_stats.get('NDVI'),
                -9999
            ))
        
        # Return -9999 if no images or if compute fails
        ndvi_val = ee.Number(ee.Algorithms.If(size.gt(0), compute(), -9999))
        
        return ee.Feature(None, {
            'date': date.format('YYYY-MM-dd'),
            'NDVI': ndvi_val
        })
    
    daily_features=daily_dates.map(process_day)
    daily_collection= ee.FeatureCollection(daily_features)

    year_data= daily_collection.getInfo()
    year_records= [f['properties'] for f in year_data['features']]
    temperature_range.extend(year_records)

    print(f'{year}, {len(year_records)} number of days')


#finally we combine to a single dataframe

df=pd.DataFrame(temperature_range)
df.to_csv('NDVI_16dayscombined1.csv', index=False)
print(f'{len(df)} saved')
    

        


2000, 366 number of days
2001, 365 number of days
2002, 365 number of days
2003, 365 number of days
2004, 366 number of days
2005, 365 number of days
2006, 365 number of days
2007, 365 number of days
2008, 366 number of days
2009, 365 number of days
2010, 365 number of days
2011, 365 number of days
2012, 366 number of days
2013, 365 number of days
2014, 365 number of days
2015, 365 number of days
2016, 366 number of days
2017, 365 number of days
2018, 365 number of days
2019, 365 number of days
2020, 366 number of days
2021, 365 number of days
2022, 365 number of days


KeyboardInterrupt: 

In [92]:
# ...existing code...
def process_day(date):
    date = ee.Date(date)
    end = date.advance(1, 'day')

    hourly_temp = ndvi_masked \
        .filterDate(date, end) \
        .select('NDVI')
    
    # Check if collection has images
    size = hourly_temp.size()
    
    def compute():
        daily_stats = hourly_temp.median()
        regional_stats = daily_stats.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=ee_object.geometry(),
            scale=250,
            maxPixels=1e9
        )
        # Safe get with unmask
        return ee.Number(ee.Algorithms.If(
            regional_stats.contains('NDVI'),
            regional_stats.get('NDVI'),
            -9999
        ))
    
    # Return -9999 if no images or if compute fails
    ndvi_val = ee.Number(ee.Algorithms.If(size.gt(0), compute(), -9999))
    
    return ee.Feature(None, {
        'date': date.format('YYYY-MM-dd'),
        'NDVI': ndvi_val
    })
# ...existing code...

In [7]:
import geemap
ee.Initialize(project='induswater')
ee_object = geemap.shp_to_ee('indus_shapefile/upperindusbd.shp')

In [ ]:
import ee
import pandas as pd
import datetime
# ...existing code...
temperature_range = []

# 1. Define the whole range as a single list (Server-side)
start_total = ee.Date('2000-02-18')
end_total = ee.Date('2026-01-01')
num_days = end_total.difference(start_total, 'day')
daily_dates = ee.List.sequence(0, num_days.subtract(1)).map(lambda d: start_total.advance(d, 'day'))

def process_day(date):
    date = ee.Date(date)
    end = date.advance(1, 'day')

    hourly_temp = ndvi_masked \
        .filterDate(date, end) \
        .select('NDVI')
    
    # Check if collection has images
    size = hourly_temp.size()
    
    def compute():
        daily_stats = hourly_temp.median()
        regional_stats = daily_stats.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=ee_object.geometry(),
            scale=250,
            maxPixels=1e9
        )
        # Safe get with unmask
        return ee.Number(ee.Algorithms.If(
            regional_stats.contains('NDVI'),
            regional_stats.get('NDVI'),
            -9999
        ))
    
    # Return -9999 if no images or if compute fails
    ndvi_val = ee.Number(ee.Algorithms.If(size.gt(0), compute(), -9999))
    
    return ee.Feature(None, {
        'date': date.format('YYYY-MM-dd'),
        'NDVI': ndvi_val
    })

features = daily_dates.map(process_day)
fc = ee.FeatureCollection(features)

# 2. EXPORT: This is the key. It bypasses the "Concurrent Aggregation" limit.
task = ee.batch.Export.table.toDrive(
collection=fc,
description='ndvicombined16days',
fileFormat='CSV',
selectors=['date', 'NDVI']
)

task.start()
print("Task started! Go to code.earthengine.google.com and check the 'Tasks' tab on the right.")

Task started! Go to code.earthengine.google.com and check the 'Tasks' tab on the right.


In [13]:
import ee
import pandas as pd

# ...existing code...
temperature_range = []

# Get all MOD13Q1 images (they're already 16-day composites)
ndvi_collection = ndvi_masked.filterDate('2000-02-18', '2026-01-01')

def process_composite(image):
    """Process each 16-day composite image"""
    # Get the actual acquisition date of this composite
    img_date = image.date()
    
    # Calculate basin-wide mean NDVI
    stats = image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=ee_object.geometry(),
        scale=250,
        maxPixels=1e9,
        tileScale=4
    )
    
    # Safe extraction
    ndvi_val = ee.Number(ee.Algorithms.If(
        stats.contains('NDVI'),
        stats.get('NDVI'),
        -9999
    ))
    
    return ee.Feature(None, {
        'date': img_date.format('YYYY-MM-dd'),
        'NDVI': ndvi_val
    })

# Map over actual images (not artificial date ranges)
features = ndvi_collection.map(process_composite)
fc = ee.FeatureCollection(features)

# Export to Drive
task = ee.batch.Export.table.toDrive(
    collection=fc,
    description='MOD13Q1_NDVI_Composites',
    fileFormat='CSV',
    selectors=['date', 'NDVI']
)

task.start()
print("Task started! Each row = one 16-day composite image")
print(f"Expected ~{ndvi_collection.size().getInfo()} records (not daily)")

Task started! Each row = one 16-day composite image
Expected ~0 records (not daily)


In [14]:
# Debug: Check original collection size
print(f"Original collection size: {ndvi_dataset.size().getInfo()}")
print(f"After masking: {ndvi_masked.size().getInfo()}")


Original collection size: 595
After masking: 595


In [15]:
# Get one sample image to test
sample = ndvi_masked.first()
print(f"Sample image bands: {sample.bandNames().getInfo()}")


Sample image bands: ['NDVI']


In [16]:
# Test if any valid pixels exist
test_stats = sample.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=ee_object.geometry(),
    scale=250,
    maxPixels=1e9
)
print(f"Test stats: {test_stats.getInfo()}")


Test stats: {'NDVI': 0.30895290657208335}


In [4]:
import ee
import pandas as pd
import datetime
import geemap
temperature_range=[]
ee.Initialize(project='induswater')
ee_object = geemap.shp_to_ee('indus_shapefile/upperindusbd.shp')

for year in range(2000,2026):
    start_date = ee.Date(f'{year}-01-01')
    if (year==2025):
        end_date = ee.Date('2026-01-01')
    else:
        end_date= ee.Date(f'{year+1}-01-01')

    num_days = end_date.difference(start_date, 'day').ceil()
    daily_dates = ee.List.sequence(0, num_days.subtract(1)).map(lambda d: start_date.advance(d, 'day'))

    def process_day(date):
        date = ee.Date(date)
        end = date.advance(1, 'day')

        hourly_temp = ee.ImageCollection('ECMWF/ERA5_LAND/HOURLY') \
            .filterDate(date, end) \
            .select('runoff')
        
        #we now have all the temps for the day, we need mean, min, max

        # daily_stats= hourly_temp.reduce(
        #     ee.Reducer.mean().combine(
        #         reducer2= ee.Reducer.min().combine(
        #             reducer2=ee.Reducer.max(),
        #             sharedInputs=True
        #         ),
        #         sharedInputs=True,
        #     )
        # )
        daily_stats= hourly_temp.sum()

        #now over to the reduceRegion
        regional_stats = daily_stats.reduceRegion(
            reducer= ee.Reducer.mean(),
            geometry= ee_object.geometry(),
            scale=11132
        )

        return ee.Feature(None,{
            'date': date.format('YYYY-MM-dd'),
            'runoff': regional_stats.get('runoff')
        })
    
    daily_features=daily_dates.map(process_day)
    daily_collection= ee.FeatureCollection(daily_features)

    year_data= daily_collection.getInfo()
    year_records= [f['properties'] for f in year_data['features']]
    temperature_range.extend(year_records)

    print(f'{year}, {len(year_records)} number of days')


#finally we combine to a single dataframe

df=pd.DataFrame(temperature_range)
df.to_csv('runoff.csv', index=False)
print(f'{len(df)} saved')
    

        


2000, 366 number of days
2001, 365 number of days
2002, 365 number of days
2003, 365 number of days
2004, 366 number of days
2005, 365 number of days
2006, 365 number of days
2007, 365 number of days
2008, 366 number of days
2009, 365 number of days
2010, 365 number of days
2011, 365 number of days
2012, 366 number of days
2013, 365 number of days
2014, 365 number of days
2015, 365 number of days
2016, 366 number of days
2017, 365 number of days
2018, 365 number of days
2019, 365 number of days
2020, 366 number of days
2021, 365 number of days
2022, 365 number of days
2023, 365 number of days
2024, 366 number of days
2025, 365 number of days
9497 saved
